In [ ]:
#============================================================
# Celda 1 — Setup y carga
#============================================================
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
from pathlib import Path

# ROOT dinámico — funciona en local, CI y cualquier entorno
ROOT = Path.cwd()
while not (ROOT / "requirements.txt").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)

OUTPUT = Path("output")
FIGS   = OUTPUT / "06_correlacion" / "figs"
TABLES = OUTPUT / "06_correlacion" / "tables"
FIGS.mkdir(parents=True, exist_ok=True)
TABLES.mkdir(parents=True, exist_ok=True)

master = pd.read_parquet(OUTPUT / "merged/master_provincia_anio.parquet")

# Subconjunto limpio (2013–2021, 42 provincias, sin nulos)
VARS = ["saldo_neto", "compraventas_vivienda", "cob_100mbps_pct",
        "poblacion_provincia", "viajeros", "tasa_atraccion", "compraventas_per_capita"]

sub = master[["anyo", "provincia"] + VARS].dropna().copy()

# Etiquetas legibles para los gráficos
LABELS = {
    "saldo_neto":             "Saldo neto migratorio",
    "compraventas_vivienda":  "Compraventas vivienda",
    "cob_100mbps_pct":        "Cobertura 100Mbps (%)",
    "poblacion_provincia":    "Población provincia",
    "viajeros":               "Viajeros (turismo)",
    "tasa_atraccion":         "Tasa de atracción",
    "compraventas_per_capita":"Compraventas per cápita",
}

plt.rcParams.update({
    "figure.dpi": 150,
    "figure.facecolor": "white",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.family": "DejaVu Sans",
    "axes.titlesize": 12,
    "axes.labelsize": 10,
})

print(f"✅ Subconjunto: {sub.shape} | años: 2013–2021 | provincias: {sub['provincia'].nunique()}")

In [ ]:
#============================================================
# Celda 2 — Matriz de correlación de Pearson
#============================================================
corr = sub[VARS].corr(method="pearson").round(3)

# Máscara triángulo superior
mask = np.triu(np.ones_like(corr, dtype=bool))

fig, ax = plt.subplots(figsize=(10, 8))

sns.heatmap(
    corr,
    mask=mask,
    annot=True,
    fmt=".2f",
    cmap="RdYlGn",
    center=0,
    vmin=-1, vmax=1,
    linewidths=0.5,
    linecolor="white",
    square=True,
    ax=ax,
    xticklabels=[LABELS[v] for v in VARS],
    yticklabels=[LABELS[v] for v in VARS],
    annot_kws={"size": 9},
)

ax.set_title("Matriz de correlación de Pearson\n(42 provincias × 2013–2021)", pad=14)
plt.xticks(rotation=35, ha="right", fontsize=9)
plt.yticks(rotation=0, fontsize=9)

plt.tight_layout()
plt.savefig(FIGS / "01_matriz_correlacion.png")
plt.show()

# Exportar
corr.to_csv(TABLES / "matriz_correlacion_pearson.csv")
print("✅ Guardado: 01_matriz_correlacion.png")

In [ ]:
#============================================================
# Celda 3 — Correlaciones con saldo_neto ordenadas
#============================================================
corr_target = (
    sub[VARS].corr(method="pearson")["saldo_neto"]
    .drop("saldo_neto")
    .sort_values()
)

fig, ax = plt.subplots(figsize=(9, 5))
colores = ["#c0392b" if v < 0 else "#27ae60" for v in corr_target]
bars = ax.barh(
    [LABELS[v] for v in corr_target.index],
    corr_target.values,
    color=colores, edgecolor="white", height=0.6
)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlim(-1, 1)
ax.set_title("Correlación de Pearson con Saldo Neto Migratorio\n(2013–2021, 42 provincias)")
ax.set_xlabel("r de Pearson")

# Anotar valores
for bar, val in zip(bars, corr_target.values):
    ax.text(val + (0.02 if val >= 0 else -0.02), bar.get_y() + bar.get_height()/2,
            f"{val:.3f}", va="center", ha="left" if val >= 0 else "right", fontsize=9)

plt.tight_layout()
plt.savefig(FIGS / "02_correlacion_con_saldo_neto.png")
plt.show()
print("✅ Guardado: 02_correlacion_con_saldo_neto.png")

In [ ]:
#============================================================
# Celda 4 — Scatter: compraventas vs saldo_neto
#============================================================
fig, ax = plt.subplots(figsize=(9, 6))

ax.scatter(
    sub["compraventas_vivienda"], sub["saldo_neto"],
    alpha=0.35, s=18, color="#2980b9", edgecolors="none"
)

# Línea de regresión
slope, intercept, r, p, se = stats.linregress(
    sub["compraventas_vivienda"], sub["saldo_neto"]
)
x_line = np.linspace(sub["compraventas_vivienda"].min(),
                     sub["compraventas_vivienda"].max(), 200)
ax.plot(x_line, slope * x_line + intercept, color="#e74c3c",
        linewidth=2, label=f"r = {r:.3f}  |  p < 0.001" if p < 0.001 else f"r = {r:.3f}  |  p = {p:.3f}")

ax.axhline(0, color="gray", linewidth=0.6, linestyle="--")
ax.set_title("Compraventas de vivienda vs Saldo neto migratorio\n(2013–2021, 42 provincias)")
ax.set_xlabel("Compraventas vivienda (nº operaciones)")
ax.set_ylabel("Saldo neto (personas)")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig(FIGS / "03_scatter_compraventas_saldo.png")
plt.show()
print(f"✅ r={r:.3f}  p={p:.4f}  slope={slope:.4f}")

In [ ]:
#============================================================
# Celda 5 — Scatter: cobertura 100Mbps vs saldo_neto
#============================================================
fig, ax = plt.subplots(figsize=(9, 6))

ax.scatter(
    sub["cob_100mbps_pct"], sub["saldo_neto"],
    alpha=0.35, s=18, color="#8e44ad", edgecolors="none"
)

slope2, intercept2, r2, p2, se2 = stats.linregress(
    sub["cob_100mbps_pct"], sub["saldo_neto"]
)
x_line2 = np.linspace(sub["cob_100mbps_pct"].min(),
                      sub["cob_100mbps_pct"].max(), 200)
ax.plot(x_line2, slope2 * x_line2 + intercept2, color="#e74c3c",
        linewidth=2, label=f"r = {r2:.3f}  |  p < 0.001" if p2 < 0.001 else f"r = {r2:.3f}  |  p = {p2:.3f}")

ax.axhline(0, color="gray", linewidth=0.6, linestyle="--")
ax.set_title("Cobertura 100Mbps vs Saldo neto migratorio\n(2013–2021, 42 provincias)")
ax.set_xlabel("Cobertura 100Mbps (%)")
ax.set_ylabel("Saldo neto (personas)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig(FIGS / "04_scatter_cobertura_saldo.png")
plt.show()
print(f"✅ r={r2:.3f}  p={p2:.4f}  slope={slope2:.4f}")

In [ ]:
#============================================================
# Celda 6 — Scatter: población vs saldo_neto (outliers etiquetados)
#============================================================
media_prov = (
    sub.groupby("provincia")[["poblacion_provincia", "saldo_neto"]]
    .mean()
    .reset_index()
)

slope3, intercept3, r3, p3, _ = stats.linregress(
    media_prov["poblacion_provincia"], media_prov["saldo_neto"]
)

fig, ax = plt.subplots(figsize=(10, 7))

ax.scatter(
    media_prov["poblacion_provincia"], media_prov["saldo_neto"],
    alpha=0.6, s=40, color="#27ae60", edgecolors="white", linewidth=0.5
)

# Línea de regresión
x_line3 = np.linspace(media_prov["poblacion_provincia"].min(),
                       media_prov["poblacion_provincia"].max(), 200)
ax.plot(x_line3, slope3 * x_line3 + intercept3, color="#e74c3c",
        linewidth=1.8, label=f"r = {r3:.3f}")

# Etiquetar top outliers por saldo absoluto
umbral = media_prov["saldo_neto"].abs().quantile(0.80)
for _, row in media_prov[media_prov["saldo_neto"].abs() > umbral].iterrows():
    ax.annotate(row["provincia"],
                xy=(row["poblacion_provincia"], row["saldo_neto"]),
                xytext=(5, 3), textcoords="offset points", fontsize=7.5,
                color="#2c3e50")

ax.axhline(0, color="gray", linewidth=0.6, linestyle="--")
ax.set_title("Población provincia vs Saldo neto migratorio (media 2013–2021)")
ax.set_xlabel("Población media (hab)")
ax.set_ylabel("Saldo neto medio (personas/año)")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x/1e6):.1f}M" if x >= 1e6 else f"{int(x/1e3)}k"))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig(FIGS / "05_scatter_poblacion_saldo.png")
plt.show()
print(f"✅ r={r3:.3f}  p={p3:.4f}")

In [ ]:
#============================================================
# Celda 7 — Evolución temporal de correlaciones año a año
#============================================================
vars_cross = ["compraventas_vivienda", "cob_100mbps_pct",
              "poblacion_provincia", "compraventas_per_capita"]

corr_anual = []
for anyo in sorted(sub["anyo"].unique()):
    df_a = sub[sub["anyo"] == anyo]
    row = {"anyo": anyo}
    for v in vars_cross:
        if df_a[v].notna().sum() > 5:
            r, _ = stats.pearsonr(df_a[v].dropna(),
                                  df_a.loc[df_a[v].notna(), "saldo_neto"])
            row[v] = round(r, 3)
        else:
            row[v] = np.nan
    corr_anual.append(row)

df_corr_anual = pd.DataFrame(corr_anual).set_index("anyo")

fig, ax = plt.subplots(figsize=(11, 5))
colores_linea = ["#2980b9", "#8e44ad", "#27ae60", "#e67e22"]
for v, color in zip(vars_cross, colores_linea):
    ax.plot(df_corr_anual.index, df_corr_anual[v],
            marker="o", markersize=5, linewidth=1.8,
            label=LABELS[v], color=color)

ax.axhline(0, color="black", linewidth=0.6, linestyle="--")
ax.axhline(0.3, color="gray", linewidth=0.4, linestyle=":", alpha=0.7)
ax.axhline(-0.3, color="gray", linewidth=0.4, linestyle=":", alpha=0.7)
ax.set_title("Evolución de correlaciones con saldo neto por año (2013–2021)")
ax.set_xlabel("Año")
ax.set_ylabel("r de Pearson")
ax.set_ylim(-1, 1)
ax.legend(fontsize=9, loc="lower left")
ax.set_xticks(df_corr_anual.index)

plt.tight_layout()
plt.savefig(FIGS / "06_correlacion_temporal.png")
plt.show()
df_corr_anual.to_csv(TABLES / "correlacion_anual.csv")
print("✅ Guardado: 06_correlacion_temporal.png")

In [ ]:
#============================================================
# Celda 8 — Correlación por segmento: urbano vs rural
#============================================================
# Clasificar provincias por población media
pob_media = sub.groupby("provincia")["poblacion_provincia"].mean()
q33 = pob_media.quantile(0.33)
q66 = pob_media.quantile(0.66)

def segmento(prov):
    p = pob_media[prov]
    if p <= q33:   return "Rural (<P33)"
    elif p <= q66: return "Intermedia (P33–P66)"
    else:          return "Urbana (>P66)"

sub["segmento"] = sub["provincia"].map(segmento)

resultados = []
for seg in ["Rural (<P33)", "Intermedia (P33–P66)", "Urbana (>P66)"]:
    df_s = sub[sub["segmento"] == seg]
    for v in ["compraventas_vivienda", "cob_100mbps_pct", "compraventas_per_capita"]:
        if df_s[v].notna().sum() > 5:
            r, p_val = stats.pearsonr(
                df_s[v].dropna(),
                df_s.loc[df_s[v].notna(), "saldo_neto"]
            )
            resultados.append({
                "segmento": seg, "variable": LABELS[v],
                "r": round(r, 3), "p_value": round(p_val, 4),
                "sig": "***" if p_val < 0.001 else "**" if p_val < 0.01 else "*" if p_val < 0.05 else "ns"
            })

df_seg = pd.DataFrame(resultados)

fig, ax = plt.subplots(figsize=(10, 5))
pivot_seg = df_seg.pivot(index="variable", columns="segmento", values="r")
pivot_seg.plot(kind="bar", ax=ax, edgecolor="white",
               color=["#c0392b", "#f39c12", "#27ae60"], width=0.65)

ax.axhline(0, color="black", linewidth=0.8)
ax.set_title("Correlación con saldo neto por segmento de provincia\n(Rural / Intermedia / Urbana)")
ax.set_xlabel("")
ax.set_ylabel("r de Pearson")
ax.set_ylim(-1, 1)
ax.legend(title="Segmento", fontsize=9)
plt.xticks(rotation=20, ha="right")

plt.tight_layout()
plt.savefig(FIGS / "07_correlacion_segmento.png")
plt.show()
df_seg.to_csv(TABLES / "correlacion_por_segmento.csv", index=False)
print("✅ Guardado: 07_correlacion_segmento.png")
print(df_seg.to_string(index=False))

In [ ]:
#============================================================
# Celda 9 — Tabla resumen de correlaciones con p-values
#============================================================
resumen_corr = []
for v in VARS:
    if v == "saldo_neto":
        continue
    r, p_val = stats.pearsonr(sub[v], sub["saldo_neto"])
    resumen_corr.append({
        "Variable":   LABELS[v],
        "r Pearson":  round(r, 4),
        "p-value":    round(p_val, 5),
        "Sig.":       "***" if p_val < 0.001 else "**" if p_val < 0.01 else "*" if p_val < 0.05 else "ns",
        "Interpretación": (
            "Correlación positiva fuerte" if r > 0.5 else
            "Correlación positiva moderada" if r > 0.3 else
            "Correlación positiva débil" if r > 0 else
            "Correlación negativa débil" if r > -0.3 else
            "Correlación negativa moderada" if r > -0.5 else
            "Correlación negativa fuerte"
        )
    })

df_resumen = pd.DataFrame(resumen_corr).sort_values("r Pearson", ascending=False)
df_resumen.to_csv(TABLES / "resumen_correlaciones.csv", index=False)

print("📊 Resumen de correlaciones con saldo_neto (2013–2021)\n")
print(df_resumen.to_string(index=False))

In [ ]:
#============================================================
# Celda 10 — Cierre y log
#============================================================
print("=" * 55)
print("🏁 06_correlacion completado")
print("=" * 55)
print(f"\n📁 Figuras  → {FIGS}")
print(f"📁 Tablas   → {TABLES}")
print("\n📈 Figuras generadas:")
for f in sorted(FIGS.glob("*.png")):
    print(f"   ✅ {f.name}")
print("\n📋 Tablas exportadas:")
for f in sorted(TABLES.glob("*.csv")):
    print(f"   ✅ {f.name}")